# Validating the dataset

In [3]:
import json
import re
import requests
from urllib.parse import urlsplit
from IPython.display import Markdown, display
import numpy
import pandas as pd

data_path = "../data"

with open(f"{data_path}/quizzes.json", "r") as f:
    quizzes = json.load(f)

with open(f"{data_path}/timeout_exceptions.json", "r") as f:
    timeout_exceptions = json.load(f)

In [2]:
quizzes[0]

{'_id': 'KVicfAwDCYWseVLEXcQ1wgoc',
 'date': '2019-10-26',
 'source_url': 'https://www.morningbrew.com/issues/2019%2F10%2F26',
 'headlines': [{'text': 'Hitman hires hitman who hires hitman who hires hitman who hires hitman who tells police',
   'url': 'https://metro.co.uk/2019/10/23/hitman-hires-hitman-hires-hitman-hires-hitman-hires-hitman-tells-police-10971438/'},
  {'text': 'Man’s body brews its own beer after yeast take over his gut microbiome',
   'url': 'https://www.newscientist.com/article/2220432-mans-body-brews-its-own-beer-after-yeast-take-over-his-gut-microbiome/'},
  {'text': 'A parking space in Hong Kong just sold for almost $1 million',
   'url': 'https://www.bloomberg.com/news/articles/2019-10-24/hong-kong-car-parking-spot-sells-for-eye-watering-1-million'},
  {'text': 'Sanders staffer fired after accidentally ordering Uber helicopter to Queens rally',
   'url': 'https://media.giphy.com/media/l0NhWLdqrEy4hgeGc/giphy.gif'}],
 'answer': 'We made up the Bernie news.'}

In [17]:
print(f"Number of quizzes: {len(quizzes)}")

# with open(f"{data_path}/exceptions.json", "r") as f:
#     exceptions = json.load(f)
# print(f"Number of exceptions: {len(exceptions)}")

Number of quizzes: 341


In [34]:
from IPython.display import display, Markdown

quiz = next(q for q in quizzes if q["date"] == "2021-08-05")
headline = quiz["headlines"][0]["text"]

print("Raw text:")
print(repr(headline))

print("\nRendered:")
display(Markdown(headline))

Raw text:
'Tech-forward Domino\'s plans to launch "\'Pizza Metaverse\''

Rendered:


Tech-forward Domino's plans to launch "'Pizza Metaverse'

Automated structural checks

In [7]:
with open(f"{data_path}/timeout_exceptions.json", "r") as f:
    timeout_exceptions = json.load(f)

"https://www.washingtonpost.com/nation/2020/03/04/jill-biden-joe-biden-protesters/" in timeout_exceptions["2020-03-07"]

True

In [18]:
date_search = re.compile(r'^\d{4}-\d{2}-\d{2}$')
dates_list = []
ids_list = []

def html_request(url):
    try:
        response = requests.get(url, timeout=10)
        if response.status_code == 404 or response.status_code == 460:
            print(f"{url} returned {response.status_code}")
    except requests.RequestException as e:
        print(f"{url} faild: {e}")

for quiz in quizzes:
    print(f"Checking quiz {quiz["date"]}")
    if not quiz["_id"]:
        print(f"{quiz["date"]} edition does not have an id")
    else:
        ids_list.append(quiz["_id"])
    if not date_search.search(quiz["date"]):
        print(f"{quiz["date"]} edition has invalid date")
    else:
        dates_list.append(quiz["date"])
    if (len(quiz["headlines"])) != 4:
        print(f"{quiz["date"]} edition does not have four headlines")
    for headline in quiz["headlines"]:
        url = headline["url"]
        if not headline["text"]:
            print(f"{quiz["date"]} edition has empty headline")
        if not url:
            print(f"{quiz["date"]} edition has empty URL")
        else:
            parts = urlsplit(url)
            if not parts.scheme:
                print(f"{url} edition has no scheme")
            if not parts.netloc:
                print(f"{url} edition has no domain")
            if not quiz["date"] in timeout_exceptions:
                html_request(url)
            else:
                if not url in timeout_exceptions[quiz["date"]]:
                    html_request(url)                
    if not any("giphy" in headline["url"] for headline in quiz["headlines"]):
        print(f"{quiz["date"]} edition has no GIF link")
    if not quiz["answer"]:
        print(f"{quiz["date"]} edition has empty answer")

ids_set = set(ids_list)
if len(ids_set) == len(ids_list):
    print("No duplicate ids detected")
else:
    print("Duplicate ids detected")
dates_set = set(dates_list)
if len(dates_set) == len(dates_list):
    print("No duplicate dates detected")
else:
    print("Duplicate dates detected")



Checking quiz 2019-10-26
Checking quiz 2019-11-02
Checking quiz 2019-11-09
Checking quiz 2019-11-16
Checking quiz 2019-11-23
Checking quiz 2019-12-07
Checking quiz 2019-12-14
Checking quiz 2019-12-21
Checking quiz 2020-01-04
Checking quiz 2020-01-18
2020-01-18 edition has no GIF link
Checking quiz 2020-01-25
Checking quiz 2020-02-01
Checking quiz 2020-02-08
Checking quiz 2020-02-15
Checking quiz 2020-02-22
Checking quiz 2020-02-29
Checking quiz 2020-03-07
Checking quiz 2020-03-14
Checking quiz 2020-03-21
https://ew.com/tv/coronavirus-masks-resident/ returned 460
Checking quiz 2020-03-28
2020-03-28 edition has no GIF link
Checking quiz 2020-04-04
Checking quiz 2020-04-18
Checking quiz 2020-04-25
https://people.com/movies/tom-hanks-gifts-corona-brand-typewriter-to-a-bullied-boy-named-corona/ returned 460
Checking quiz 2020-05-02
Checking quiz 2020-05-09
Checking quiz 2020-05-16
Checking quiz 2020-05-23
Checking quiz 2020-05-30
Checking quiz 2020-06-06
Checking quiz 2020-06-13
Checking qu

Automated anomaly detection

In [12]:
quizzes[1]

{'_id': 'z9gwGZ5GPFVVhEwb6cJdPxos',
 'date': '2019-11-02',
 'source_url': 'https://www.morningbrew.com/issues/2019%2F11%2F02',
 'headlines': [{'text': 'Leading maggot farmer to expand from Cape Town to California',
   'url': 'https://www.bloomberg.com/news/articles/2019-10-31/leading-maggot-farmer-to-expand-from-cape-town-to-california'},
  {'text': 'Riding wave of support for foie gras ban, de Blasio re-enters presidential race',
   'url': 'https://media.giphy.com/media/PoLEEuKQT7FnnJRiAA/giphy.gif'},
  {'text': 'Migrating Russian eagles run up huge data roaming charges',
   'url': 'https://www.bbc.com/news/world-europe-50180781'},
  {'text': 'Horsepower, literally: Finnish horse show runs on manure',
   'url': 'https://apnews.com/3ad8a6b9b1cb4100a9db1b2e406ad14d'}],
 'answer': 'Bill de Blasio is not re-entering the presidential race.'}

In [37]:
anomalous_characters = ["[", "]", "\n", "\r", "\t" "\xa0", "]("]
question_numbers = ["1.", "2.", "3.", "4."]


for quiz in quizzes:
    headlines = []
    print(f"Checking quiz {quiz["date"]}")
    for headline in quiz["headlines"]:
        text = headline["text"]
        headlines.append(text)
        url = headline["url"]
        if len(text) > 120:
            print(f"Long headline text detected in {quiz["date"]} edition: {text}")
        if len(text) < 40:
            print(f"Short headline text detected in {quiz["date"]} edition: {text}")
        if any(character in text for character in anomalous_characters):
            print(f"Anomalous characters detected in {quiz["date"]} edition headline: {repr(text)}")
        if any(text.startswith(question_number) for question_number in question_numbers):
            print(f"{quiz["date"]} edition headline starts with question number: {repr(text)}")
        if "https:" in text:
            print(f"{quiz["date"]} edition headline contains 'https:': {text}")
        if text.startswith(" "):
            print(f"{quiz["date"]} edition headline starts with whitespace: {text}")
        if text.endswith(" "):
            print(f"{quiz["date"]} edition headline ends with whitespace: {text}")
        if text.startswith("“") and text.endswith("”"):
            print(f"{quiz["date"]} edition headline starts and ends with curly quotation marks: {text}")
        if text.startswith("\"") and text.endswith("\""):
            print(f"{quiz["date"]} edition headline starts and ends with vertical quotation marks: {text}")
        if text.startswith("“") and text.endswith("\""):
            print(f"{quiz["date"]} edition headline starts with curly quotation marks and ends with vertical quotation marks: {text}")
        if text.startswith("\"") and text.endswith("”"):
            print(f"{quiz["date"]} edition headline starts with vertical quotation marks and ends with curly quotation marks: {text}")
        if text.startswith("”") and text.endswith("”"):
            print(f"{quiz["date"]} edition headline starts and ends with opening quotation marks: {text}")
        if text.startswith("“") and text.endswith("“"):
            print(f"{quiz["date"]} edition headline starts and ends with closing quotation marks: {text}")
        if "morningbrew" in url or "morning_brew" in url:
            print(f"{quiz["date"]} edition URL contains 'morningbrew' or 'morning_brew': {text}")
        if re.search(r"\S_\S", text):
            print(f"{quiz['date']} edition contains underscore without spaces on both sides: {repr(text)}")
        if re.search(r"(^|\s)[“\"]\s|\s[”\"]($|\s)", text):
            print(f"{quiz['date']} edition contains whitespace on both sides of a quotation mark: {repr(text)}")
    if len(set(headlines)) != 4:
        print(f"{quiz["date"]} edition contains duplicate headlines: {quiz}")
    answer = quiz["answer"]
    if "answer" in answer:
        print(f"{quiz["date"]} edition answer contains 'answer': {quiz["answer"]}")
    if len(answer) > 110:
        print(f"Long answer text detected in {quiz["date"]} edition: {answer}")
    if len(answer) < 27:
        print(f"Short answer text detected in {quiz["date"]} edition: {answer}")

Checking quiz 2019-10-26
Checking quiz 2019-11-02
Checking quiz 2019-11-09
Checking quiz 2019-11-16
Short headline text detected in 2019-11-16 edition: Twinkies cereal is now in the works
Checking quiz 2019-11-23
Short headline text detected in 2019-11-23 edition: Why the world is running out of sand
Checking quiz 2019-12-07
Checking quiz 2019-12-14
Short headline text detected in 2019-12-14 edition: ‘My boss lets us book hangover days’
Checking quiz 2019-12-21
Checking quiz 2020-01-04
Checking quiz 2020-01-18
Long answer text detected in 2020-01-18 edition: The Men's Wearhouse story isn't real...but you should read the [Onion article](https://www.theonion.com/cash-strapped-men-s-wearhouse-now-offering-free-measure-1840982358?utm_source=morning_brew), it's hilarious.
Checking quiz 2020-01-25
Checking quiz 2020-02-01
Checking quiz 2020-02-08
Checking quiz 2020-02-15
Short headline text detected in 2020-02-15 edition: Psychedelics on the stock exchange?
Checking quiz 2020-02-22
Checking 

In [38]:
for quiz in quizzes:
    headlines = []
    print(f"Checking quiz {quiz["date"]}")
    for headline in quiz["headlines"]:
        text = headline["text"]
        headlines.append(text)
    if len(set(headlines)) != 4:
        print(f"{quiz["date"]} edition contains duplicate headlines: {quiz}")

Checking quiz 2019-10-26
Checking quiz 2019-11-02
Checking quiz 2019-11-09
Checking quiz 2019-11-16
Checking quiz 2019-11-23
Checking quiz 2019-12-07
Checking quiz 2019-12-14
Checking quiz 2019-12-21
Checking quiz 2020-01-04
Checking quiz 2020-01-18
Checking quiz 2020-01-25
Checking quiz 2020-02-01
Checking quiz 2020-02-08
Checking quiz 2020-02-15
Checking quiz 2020-02-22
Checking quiz 2020-02-29
Checking quiz 2020-03-07
Checking quiz 2020-03-14
Checking quiz 2020-03-21
Checking quiz 2020-03-28
Checking quiz 2020-04-04
Checking quiz 2020-04-18
Checking quiz 2020-04-25
Checking quiz 2020-05-02
Checking quiz 2020-05-09
Checking quiz 2020-05-16
Checking quiz 2020-05-23
Checking quiz 2020-05-30
Checking quiz 2020-06-06
Checking quiz 2020-06-13
Checking quiz 2020-06-20
Checking quiz 2020-06-27
Checking quiz 2020-07-11
Checking quiz 2020-07-18
Checking quiz 2020-07-25
Checking quiz 2020-08-01
Checking quiz 2020-08-08
Checking quiz 2020-08-15
Checking quiz 2020-08-22
Checking quiz 2020-08-29


In [8]:
q_nums = ["1.", "2."]
string = "1. foo"
any(string.startswith(number) for number in q_nums)

True

In [ ]:
len(quizzes[0]["headlines"][0]["text"])

89

Manual sampling and validation

In [7]:
quizzes[1]


{'_id': 'z9gwGZ5GPFVVhEwb6cJdPxos',
 'date': '2019-11-02',
 'source_url': 'https://www.morningbrew.com/issues/2019%2F11%2F02',
 'headlines': [{'text': 'Leading maggot farmer to expand from Cape Town to California',
   'url': 'https://www.bloomberg.com/news/articles/2019-10-31/leading-maggot-farmer-to-expand-from-cape-town-to-california'},
  {'text': 'Riding wave of support for foie gras ban, de Blasio re-enters presidential race',
   'url': 'https://media.giphy.com/media/PoLEEuKQT7FnnJRiAA/giphy.gif'},
  {'text': 'Migrating Russian eagles run up huge data roaming charges',
   'url': 'https://www.bbc.com/news/world-europe-50180781'},
  {'text': 'Horsepower, literally: Finnish horse show runs on manure',
   'url': 'https://apnews.com/3ad8a6b9b1cb4100a9db1b2e406ad14d'}],
 'answer': 'Bill de Blasio is not re-entering the presidential race.'}

In [39]:
for index in numpy.linspace(0, len(quizzes) - 1, num=30):
    index = round(index)
    display(quizzes[index]["date"])
    for headline in quizzes[index]["headlines"]:
        display(Markdown(headline["text"]))
        print(headline["url"])

'2019-10-26'

Hitman hires hitman who hires hitman who hires hitman who hires hitman who tells police

https://metro.co.uk/2019/10/23/hitman-hires-hitman-hires-hitman-hires-hitman-hires-hitman-tells-police-10971438/


Man’s body brews its own beer after yeast take over his gut microbiome

https://www.newscientist.com/article/2220432-mans-body-brews-its-own-beer-after-yeast-take-over-his-gut-microbiome/


...

https://www.nytimes.com/2026/10/06/us/florida-keys-cocaine.html


ChatGPT is adding real cartoonists’ signatures to fake New Yorker cartoons

https://www.niemanlab.org/2026/10/chatgpt-is-adding-real-cartoonists-signatures-to-fake-new-yorker-cartoons/


Every lip product is also a cheek product, now. It’s because of this woman and her cat.

https://giphy.com/gifs/bailando-ai-dance-2banana1-tphCApwvdtC1VJabZ1


In [26]:
response = requests.get("https://www.foodandwine.com/motorcycle-that-runs-on-beer-7498431", timeout=10)
response.status_code

460

In [32]:
with open(f"{data_path}/manual_quiz_additions.json", "r") as f:
    manual_additions = json.load(f)

for headline in manual_additions[3]["headlines"]:
        display(Markdown(headline["text"]))
        print(headline["url"])

Italian designer creates the 'trikini,' a bikini set with matching face mask

https://abcnews.com/GMA/Style/designer-launches-trikini-beachwear-design-complete-matching-bikini/story?id=70657609


These face masks come with a straw hole for sipping cocktails

https://nypost.com/2020/05/12/these-face-masks-come-with-straw-hole-for-sipping-cocktails/


Gucci halts production of its $6,000 masks as customers complain about breathing through the leather

https://media.giphy.com/media/LpWmpM78HHoWOeQJIn/giphy.gif


Men are less likely than women to wear face masks because they see it as 'a sign of weakness,' a survey found

https://www.businessinsider.com/men-less-likely-to-wear-masks-because-not-cool-study-2020-5


Outstanding issues

In [4]:
with open(f"{data_path}/known_issues_outstanding.json", "r") as f:
    outstanding_issues = json.load(f)

# df = pd.DataFrame.from_dict(outstanding_issues, orient="index")

rows = []

for date, issue in outstanding_issues.items():
    for item in issue["edition_issues"] + issue["headline_issues"]:
        rows.append({
            "date": date,
            "type": item["type"],
            "description": item["description"]
        })

df = pd.DataFrame(rows)
pd.set_option("display.max_colwidth", None)
display(df)

""


Diffs between old and new datasets

In [6]:
import json, difflib

OLD, NEW = f"{data_path}/old/ml_dataset.json", f"{data_path}/ml_dataset.json"

old = {q["date"]: q for q in json.load(open(OLD, encoding="utf-8"))}
new = {q["date"]: q for q in json.load(open(NEW, encoding="utf-8"))}

def vis(s):
    return "".join(c if c.isascii() and c.isprintable() else f"\\u{ord(c):04x}" for c in s)

def inline(a, b):
    out = []
    for tag, i1, i2, j1, j2 in difflib.SequenceMatcher(None, a, b, autojunk=False).get_opcodes():
        if tag == "equal":
            out.append(a[i1:i2])
        else:
            if i2 > i1: out.append("[-" + vis(a[i1:i2]) + "-]")
            if j2 > j1: out.append("{+" + vis(b[j1:j2]) + "+}")
    return "".join(out)

changed = 0
lines = []
for d in sorted(old.keys() | new.keys()):
    if d not in old: lines.append(f"{d}: ADDED"); changed += 1; continue
    if d not in new: lines.append(f"{d}: REMOVED"); changed += 1; continue
    o, n = old[d], new[d]
    quiz_lines = []
    if o["answer"] != n["answer"]:
        quiz_lines.append(f"  answer: {o['answer']} -> {n['answer']}")
    if len(o["headlines"]) != len(n["headlines"]):
        quiz_lines.append(f"  headline count: {len(o['headlines'])} -> {len(n['headlines'])}")
    else:
        for i, (a, b) in enumerate(zip(o["headlines"], n["headlines"])):
            if a != b:
                quiz_lines.append(f"  [{i}] {inline(a, b)}")
    if quiz_lines:
        changed += 1
        lines.append(d)
        lines.extend(quiz_lines)

lines.append(f"\nQuizzes changed: {changed} of {len(new)}")
print("\n".join(lines))

2020-02-08
  [0] Antarctica logs hottest temperature on record with a reading of 18.3[-\u00c2-]°C (65[-\u00c2-]°F)
2020-04-04
  [2] [-\u201d-]From toilet paper to face mask in 5 easy steps[-\u201d-]
2020-09-05
  [0] Man aims to see _Tenet_{+ +}120 times for Guinness record
2021-05-13
  [2] IHOP turned away{+ +}Adam Sandler. Now it's trying to make it up to him
2021-06-24
  [2] {+42+},000 pounds of missing pistachios leads to possible illegal pistachio operation
2021-08-05
  [0] Tech-forward Domino's plans to launch [-"-]'Pizza Metaverse'
2022-02-03
  [2] {+4+}0-person brawl breaks out at Golden Corral over steak
2022-06-30
  [1] New Wimbledon documentary has interviews with former pros, journalists, and[-\u00e2\u20ac\u00a6-]{+\u2026 +}Dick Cheney
  [3] Man loses USB flash drive with data on entire city[-\u00e2\u20ac\u2122-]{+\u2019+}s residents after night out
2023-01-12
  [2] Prince Harry’s new book _Spare_{+ +}reveals his 12th birthday party theme was ‘office supplies’
2023-02-02
  [

Checking for different types of quotes as well as trailing full stops and backslashes

In [5]:
import json, math, re, collections

data = json.load(open(f"{data_path}/ml_dataset.json", encoding="utf-8"))
rows = [(q["date"], i, h, i == q["answer"]) for q in data for i, h in enumerate(q["headlines"])]

CURLY = "\u2018\u2019\u201c\u201d"
feats = {
    "straight quote/apostrophe": lambda s: "'" in s or '"' in s,
    "curly quote/apostrophe": lambda s: any(c in s for c in CURLY),
    "straight AND curly": lambda s: ("'" in s or '"' in s) and any(c in s for c in CURLY),
    "trailing full stop": lambda s: s.rstrip("\"'\u2019\u201d").endswith("."),
    "backslash": lambda s: "\\" in s,
    "non-ASCII": lambda s: not s.isascii(),
}
for name, f in feats.items():
    hit = [r for r in rows if f(r[2])]
    n, k = len(hit), sum(r[3] for r in hit)
    sd = math.sqrt(n * 0.25 * 0.75)
    print(f"{name:28s} n={n:4d} on_answer={k:3d} expected={n/4:6.1f} z={(k - n/4)/sd if sd else 0:+.1f}")

for d, i, h, _ in rows:
    p = []
    if h != h.strip(): p.append("leading/trailing space")
    if "  " in h: p.append("double space")
    if "\u00a0" in h: p.append("NBSP")
    if re.match(r"[a-z,.\-)\]]", h): p.append("starts lowercase/punct")
    if re.search("[\u00c2\u00e2\u00c3]", h): p.append("mojibake")
    if h.count("_") % 2: p.append("unbalanced underscore")
    if p: print(d, i, p, h[:60])

print([(h, c) for h, c in collections.Counter(r[2] for r in rows).items() if c > 1])

straight quote/apostrophe    n=  85 on_answer= 13 expected=  21.2 z=-2.1
curly quote/apostrophe       n= 431 on_answer=122 expected= 107.8 z=+1.6
straight AND curly           n=   8 on_answer=  3 expected=   2.0 z=+0.8
trailing period              n=  41 on_answer=  8 expected=  10.2 z=-0.8
backslash                    n=   4 on_answer=  2 expected=   1.0 z=+1.2
non-ASCII                    n= 452 on_answer=125 expected= 113.0 z=+1.3
2020-06-13 3 ['NBSP'] _Fortnite_ glitch lets players build customized exploding dr
[('Hitman hires hitman who hires hitman who hires hitman who hires hitman who tells police', 2)]
